In [ ]:
!pip install torch==2.6.0 --quiet
!pip install pandas==2.2.3 --quiet
!pip install matplotlib==3.10.1 --quiet
!pip install numpy==2.2.3 --quiet
import torch
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import time

In [ ]:
GLOBAL_SEED = 7 #изменять данный seed
torch.manual_seed(GLOBAL_SEED)
torch.set_num_threads(4)

ds = [2, 5, 10, 20]
ms = [32, 64, 128, 256, 512, 1024]
n_train_base = 50000
n_test = 10000

def rff_features(X, omega, b):
    return np.sqrt(2.0 / omega.shape[0]) * torch.cos(X @ omega.T + b)

def ridge_regression(Z, y, lam):
    n, m = Z.shape
    I = torch.eye(m, device=Z.device, dtype=Z.dtype)
    return torch.linalg.solve(Z.T @ Z + n * lam * I, Z.T @ y)

def f_radial(X):
    d = X.shape[1]
    c = 0.5 * torch.ones(d)
    return torch.exp(-0.5 * torch.sum((X - c)**2, dim=1))

def f_ridge_sum(X):
    d = X.shape[1]
    a1 = torch.tensor([1, 1] + [0.5]*(d-2), dtype=torch.float32)[:d]
    a2 = torch.tensor([0.5, 1, 1] + [0.5]*(d-3), dtype=torch.float32)[:d]
    a3 = torch.tensor([1, 0.5, 1, 1] + [1]*(d-4), dtype=torch.float32)[:d]
    a1 = a1 / torch.norm(a1)
    a2 = a2 / torch.norm(a2)
    a3 = a3 / torch.norm(a3)
    b1,b2,b3 = 0.1,0.7,0.3
    return torch.tanh(X @ a1 + b1) + torch.sin(X @ a2 + b2) + torch.cos(X @ a3 + b3)

def f_fourier_narrow(X):
    d = X.shape[1]
    w1 = torch.ones(d)
    w2 = torch.tensor([2,1.5] + [1]*(d-2))[:d]
    return torch.cos(X @ w1) + 0.5*torch.sin(X @ w1) + 0.8*torch.cos(X @ w2) - 0.3*torch.sin(X @ w2)

def f_fourier_noisy(X):
    d = X.shape[1]
    w1 = torch.ones(d)
    w2 = torch.ones(d)*10
    return torch.cos(X @ w1) + 0.5*torch.sin(X @ w1) + 0.8*torch.cos(X @ w2) - 0.3*torch.sin(X @ w2)

names = ["Radial", "Ridge sum", "Fourier narrow", "Fourier noisy"]
funcs = [f_radial, f_ridge_sum, f_fourier_narrow, f_fourier_noisy]

results = []

for d in ds:
    gen_train = torch.Generator().manual_seed(GLOBAL_SEED + 1000*d)
    gen_test  = torch.Generator().manual_seed(GLOBAL_SEED + 2000*d)
    n_train = n_train_base * d
    x_train = torch.rand(n_train, d, generator=gen_train)
    x_test  = torch.rand(n_test, d, generator=gen_test)

    for f, name in zip(funcs, names):
        y_train = f(x_train)
        y_test  = f(x_test)

        for m in ms:
            ridge_lambda = 1 / (m)
            gen_rff = torch.Generator().manual_seed(GLOBAL_SEED + 1000*d + m)
            sigma = 0.05 if name == "Fourier noisy" else np.sqrt(d)

            omega = torch.randn(m, d, generator=gen_rff) / sigma
            b = 2 * np.pi * torch.rand(m, generator=gen_rff)

            start_time = time.time()
            phi_train = rff_features(x_train, omega, b)
            phi_test = rff_features(x_test, omega, b)

            theta = ridge_regression(phi_train, y_train, ridge_lambda)
            train_time = time.time() - start_time

            y_pred = phi_test @ theta
            l2_error = torch.sqrt(torch.mean((y_pred - y_test) ** 2)).item()

            results.append({
                "function": name,
                "d": d,
                "m": m,
                "L2_error": l2_error,
                "train_time_sec": train_time
            })

            print(f" {name:14} d={d:2d}, m={m:4d}, L2={l2_error:.8f}, time={train_time:.3f}s")

df = pd.DataFrame(results)

for name in names:
    plt.figure(figsize=(10,6))
    subset = df[df["function"] == name]
    for d in ds:
        sub_d = subset[subset["d"]==d]
        plt.plot(sub_d["m"], sub_d["L2_error"], marker='o', label=f"d={d}")
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel("Число признаков $m$")
    plt.ylabel("Тестовая $L^2$-ошибка")
    plt.title(f"RFF + Ridge: {name}")
    plt.grid(True, which="both", ls="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()
results = []